In [ ]:
ufs/ufs.ipynb

Dataset Shape: (10000, 55)


In [17]:
import os
import time

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import kneighbors_graph
from sklearn.cluster import KMeans
from sklearn.linear_model import Lasso
from sklearn.feature_selection import mutual_info_classif

from scipy.linalg import eigh


# ============================================================
# OPTIONAL TORCH SUPPORT FOR CAE
# ============================================================

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False


# ============================================================
# NOISE HELPERS
# ============================================================

NOISE_LABELS = ("noise_", "heavily_shuffled_")


def is_noise(col):
    if "noise" in str(col) or "shuffle" in str(col) or "spurious" in str(col):
        return True
    return False


def count_noise(cols):
    return sum(is_noise(c) for c in cols)


# ============================================================
# PREPROCESSING
# ============================================================


def prepare_dataframe(df):
    df = df.copy()

    num_cols = df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    cat_cols = [c for c in df.columns if c not in num_cols]

    groups = {}
    parts = []
    offset = 0

    if num_cols:
        imp = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        arr = imp.fit_transform(df[num_cols])
        arr = scaler.fit_transform(arr)

        parts.append(arr)

        for i, c in enumerate(num_cols):
            groups[c] = [offset + i]

        offset += len(num_cols)

    if cat_cols:
        cat = df[cat_cols].astype(str).fillna("__missing__")

        try:
            ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        except TypeError:
            ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")

        carr = ohe.fit_transform(cat)
        parts.append(carr)

        for j, c in enumerate(cat_cols):
            nlev = len(ohe.categories_[j])
            groups[c] = list(range(offset, offset + nlev))
            offset += nlev

    X = np.hstack(parts)

    return X, groups


# ============================================================
# SCORE AGGREGATION
# ============================================================


def aggregate_scores(scores, groups, larger_better=True):
    out = {}
    scores = np.asarray(scores).ravel()

    for col, idxs in groups.items():
        vals = scores[idxs]
        out[col] = float(np.max(vals) if larger_better else np.min(vals))

    return out


# ============================================================
# AUTO-SELECTION HELPERS
# ============================================================


def _elbow_cut(values: np.ndarray, larger_better: bool) -> int:
    n = len(values)

    if n == 1:
        return 1

    sorted_vals = np.sort(values)[::-1] if larger_better else np.sort(values)

    lo, hi = sorted_vals.min(), sorted_vals.max()

    if hi == lo:
        return n

    normed = (sorted_vals - lo) / (hi - lo)

    if n == 2:
        return 1

    d2 = np.diff(normed, n=2)

    elbow_pos = int(np.argmax(np.abs(d2))) + 1

    k = elbow_pos + 1

    return max(1, min(k, n))



def _select_top_k(scores: np.ndarray, k: int, larger_better: bool):
    if larger_better:
        return np.argsort(scores)[::-1][:k]
    else:
        return np.argsort(scores)[:k]



def _auto_select(scores, k, larger_better):
    if k is None:
        k = _elbow_cut(scores, larger_better)

    return _select_top_k(scores, k, larger_better)


# ============================================================
# GRAPH CONSTRUCTION
# ============================================================


def build_similarity_graph(X, k=5, sigma=1.0):
    knn = kneighbors_graph(
        X,
        n_neighbors=k,
        mode="distance",
        include_self=False,
    )

    D = knn.toarray()

    W = np.exp(-(D ** 2) / (2 * sigma ** 2))

    W[D == 0] = 0

    W = np.maximum(W, W.T)

    return W


# ============================================================
# VARIANCE SCORE
# ============================================================


def variance_score(X, k=None):
    scores = np.var(X, axis=0)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# LAPLACIAN SCORE
# ============================================================


def laplacian_score(X, W, k=None):
    D_mat = np.diag(W.sum(axis=1))
    L = D_mat - W
    d_diag = np.diag(D_mat)

    scores = []

    for j in range(X.shape[1]):
        f = X[:, j]

        f = f - (f @ d_diag) / d_diag.sum()

        num = f @ L @ f
        den = f @ D_mat @ f + 1e-12

        scores.append(num / den)

    scores = np.array(scores)

    selected = _auto_select(scores, k, larger_better=False)

    return scores, selected


# ============================================================
# SPEC
# ============================================================


def spec_score(X, W, k=None):
    D_mat = np.diag(W.sum(axis=1))
    L = D_mat - W

    eigvals, eigvecs = eigh(L, D_mat)

    scores = []

    for j in range(X.shape[1]):
        f = X[:, j]

        coeff = np.abs(eigvecs.T @ f)

        scores.append(np.sum(coeff[1:] / (eigvals[1:] + 1e-12)))

    scores = np.array(scores)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# MCFS
# ============================================================


def mcfs_score(X, W, n_clusters=5, k=None):
    D_mat = np.diag(W.sum(axis=1))
    L = D_mat - W

    eigvals, eigvecs = eigh(L, D_mat)

    U = eigvecs[:, 1:n_clusters + 1]

    scores = []

    for j in range(X.shape[1]):
        f = X[:, j]

        score = np.sum(np.abs(np.corrcoef(f, U.T)[0, 1:]))

        scores.append(score)

    scores = np.array(scores)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# NDFS
# ============================================================


def ndfs_score(X, W, n_clusters=5, alpha=1.0, beta=1.0, k=None):
    D = np.diag(W.sum(axis=1))
    L = D - W

    eigvals, eigvecs = eigh(L, D)

    Y = eigvecs[:, 1:n_clusters + 1]

    p = X.shape[1]

    scores = np.zeros(p)

    for j in range(p):
        f = X[:, j]

        corr = np.sum(np.abs(np.corrcoef(f, Y.T)[0, 1:]))

        sparsity = np.linalg.norm(f, ord=1)

        scores[j] = corr / (alpha * sparsity + beta)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# RSR
# ============================================================


def rsr_score(X, W, n_clusters=5, alpha=0.01, k=None):
    D = np.diag(W.sum(axis=1))
    L = D - W

    eigvals, eigvecs = eigh(L, D)

    Y = eigvecs[:, 1:n_clusters + 1]

    p = X.shape[1]

    scores = np.zeros(p)

    for i in range(Y.shape[1]):
        target = Y[:, i]

        model = Lasso(alpha=alpha, max_iter=2000)
        model.fit(X, target)

        scores += np.abs(model.coef_)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# AGUFS
# ============================================================


def agufs_score(X, W, gamma=1.0, k=None):
    D = np.diag(W.sum(axis=1))
    L = D - W

    p = X.shape[1]

    scores = np.zeros(p)

    for j in range(p):
        f = X[:, j]

        smoothness = (f.T @ L @ f) / (f.T @ D @ f + 1e-12)

        variance = np.var(f)

        scores[j] = variance / (smoothness + gamma)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# UDFS-LIKE
# ============================================================


def udfs_like_score(X, alpha=0.01, k=None):
    p = X.shape[1]

    scores = np.zeros(p)

    for j in range(p):
        y = X[:, j]

        X_other = np.delete(X, j, axis=1)

        model = Lasso(alpha=alpha, max_iter=2000)

        model.fit(X_other, y)

        scores[j] = np.sum(np.abs(model.coef_))

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# CORRELATION UNIQUENESS
# ============================================================


def correlation_uniqueness_score(X, k=None):
    corr = np.corrcoef(X.T)

    corr = np.nan_to_num(corr)

    scores = []

    for i in range(corr.shape[0]):
        mean_abs_corr = np.mean(np.abs(np.delete(corr[i], i)))

        scores.append(1.0 / (mean_abs_corr + 1e-8))

    scores = np.array(scores)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# mRMR
# ============================================================


def mrmr_score(X, n_clusters=5, k=None):
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=0)

    labels = km.fit_predict(X)

    p = X.shape[1]

    relevance = np.array([
        mutual_info_classif(X[:, [j]], labels)[0]
        for j in range(p)
    ])

    corr = np.abs(np.corrcoef(X.T))

    redundancy = np.zeros(p)

    for j in range(p):
        redundancy[j] = np.mean(np.delete(corr[j], j))

    scores = relevance - redundancy

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# JMI
# ============================================================


def jmi_score(X, n_clusters=5, k=None):
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=0)

    labels = km.fit_predict(X)

    p = X.shape[1]

    relevance = np.array([
        mutual_info_classif(X[:, [j]], labels)[0]
        for j in range(p)
    ])

    corr = np.abs(np.corrcoef(X.T))

    scores = np.zeros(p)

    for j in range(p):
        redundancy = np.mean(np.delete(corr[j], j))

        scores[j] = relevance[j] / (redundancy + 1e-12)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# CMIM
# ============================================================


def cmim_score(X, n_clusters=5, k=None):
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=0)

    labels = km.fit_predict(X)

    p = X.shape[1]

    relevance = np.array([
        mutual_info_classif(X[:, [j]], labels)[0]
        for j in range(p)
    ])

    corr = np.abs(np.corrcoef(X.T))

    scores = np.zeros(p)

    for j in range(p):
        redundancy = np.max(np.delete(corr[j], j))

        scores[j] = relevance[j] - redundancy

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# INF-FS
# ============================================================


def inf_fs_score(X, alpha=0.5, k=None):
    p = X.shape[1]

    corr = np.abs(np.corrcoef(X.T))

    np.fill_diagonal(corr, 0)

    stds = np.std(X, axis=0)

    std_matrix = np.outer(stds, stds)

    A = alpha * corr + (1 - alpha) * std_matrix

    I = np.eye(p)

    S = np.linalg.inv(I - 0.9 * A)

    scores = np.sum(S, axis=1)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# CLUSTER DISPERSION
# ============================================================


def cluster_dispersion_score(X, n_clusters=5, random_state=0, k=None):
    km = KMeans(
        n_clusters=n_clusters,
        n_init=10,
        random_state=random_state,
    )

    labels = km.fit_predict(X)

    p = X.shape[1]

    scores = np.zeros(p)

    global_mean = X.mean(axis=0)

    for j in range(p):
        between = 0.0
        within = 0.0

        for c in np.unique(labels):
            idx = labels == c

            xc = X[idx, j]

            mu_c = xc.mean()

            n_c = idx.sum()

            between += n_c * (mu_c - global_mean[j]) ** 2

            within += np.sum((xc - mu_c) ** 2)

        scores[j] = between / (within + 1e-12)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# PFA
# ============================================================


def principal_feature_analysis_score(X, n_components=None, k=None):
    if n_components is None:
        n_components = min(5, X.shape[1])

    cov = np.cov(X.T)

    eigvals, eigvecs = np.linalg.eigh(cov)

    idx = np.argsort(eigvals)[::-1][:n_components]

    V = eigvecs[:, idx]

    scores = np.sum(V ** 2, axis=1)

    selected = _auto_select(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# FEATURE CLUSTER REPRESENTATIVE
# ============================================================


def feature_cluster_representative_score(
    X,
    n_feature_clusters=5,
    k=None,
):
    F = X.T

    n_clusters_eff = min(n_feature_clusters, F.shape[0])

    km = KMeans(
        n_clusters=n_clusters_eff,
        n_init=10,
        random_state=0,
    )

    labels = km.fit_predict(F)

    centers = km.cluster_centers_

    scores = np.zeros(F.shape[0])

    for i in range(F.shape[0]):
        c = labels[i]

        dist = np.linalg.norm(F[i] - centers[c])

        scores[i] = 1.0 / (dist + 1e-12)

    if k is None:
        selected = []

        for c in range(n_clusters_eff):
            members = np.where(labels == c)[0]

            best = members[np.argmax(scores[members])]

            selected.append(int(best))

        selected = np.array(selected)

    else:
        selected = _select_top_k(scores, k, larger_better=True)

    return scores, selected


# ============================================================
# CONCRETE AUTOENCODER
# ============================================================


if TORCH_AVAILABLE:

    class ConcreteSelector(nn.Module):
        def __init__(self, input_dim, k):
            super().__init__()
            self.logits = nn.Parameter(torch.randn(k, input_dim))

        def forward(self, x, temperature=0.1):
            weights = torch.softmax(self.logits / temperature, dim=1)
            selected = torch.matmul(x, weights.T)
            return selected, weights


    class CAEModel(nn.Module):
        def __init__(self, input_dim, k, hidden_dim=128):
            super().__init__()

            self.selector = ConcreteSelector(input_dim, k)

            self.decoder = nn.Sequential(
                nn.Linear(k, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, input_dim),
            )

        def forward(self, x):
            selected, weights = self.selector(x)
            recon = self.decoder(selected)
            return recon, weights


    def cae_score(X, k_select=10, epochs=50, lr=1e-3):
        X_tensor = torch.tensor(X, dtype=torch.float32)

        model = CAEModel(X.shape[1], k_select)

        optimizer = optim.Adam(model.parameters(), lr=lr)

        loss_fn = nn.MSELoss()

        for epoch in range(epochs):
            optimizer.zero_grad()

            recon, weights = model(X_tensor)

            loss = loss_fn(recon, X_tensor)

            loss.backward()

            optimizer.step()

        final_weights = model.selector.logits.detach().cpu().numpy()

        scores = np.max(final_weights, axis=0)

        selected = np.argsort(scores)[::-1][:k_select]

        return scores, selected


# ============================================================
# X INDICES TO ORIGINAL COLUMNS
# ============================================================


def _x_indices_to_columns(selected_x_indices, groups):
    selected_set = set(int(i) for i in selected_x_indices)

    return [
        col
        for col, idxs in groups.items()
        if any(i in selected_set for i in idxs)
    ]


# ============================================================
# MAIN BENCHMARK
# ============================================================


def run_ufs_benchmark(
    df,
    benchmark,
    target_column,
    n_keep=None,
    graph_k=5,
    graph_sigma=1.0,
    mcfs_clusters=5,
    udfs_alpha=0.01,
    show_progress=True,
):
    total_start = time.perf_counter()

    if show_progress:
        print("\n[1/3] Preparing dataframe...")

    prep_start = time.perf_counter()

    X, groups = prepare_dataframe(df)

    prep_time = time.perf_counter() - prep_start

    if show_progress:
        print(f"Done in {prep_time:.3f} sec")

    if show_progress:
        print("\n[2/3] Building similarity graph...")

    graph_start = time.perf_counter()

    W = build_similarity_graph(X, k=graph_k, sigma=graph_sigma)

    graph_time = time.perf_counter() - graph_start

    if show_progress:
        print(f"Done in {graph_time:.3f} sec")

    methods = {
        "variance_score": (lambda: variance_score(X, k=n_keep),True,),
        "laplacian_score": (lambda: laplacian_score(X, W, k=n_keep),False,),
        "spec": (lambda: spec_score(X, W, k=n_keep),True,),
        "mcfs": (lambda: mcfs_score(X, W, n_clusters=mcfs_clusters, k=n_keep),True,),
        "ndfs": (lambda: ndfs_score(X, W, n_clusters=mcfs_clusters, k=n_keep),True,),
        "rsr": (lambda: rsr_score(X, W, n_clusters=mcfs_clusters, k=n_keep),True,),
        "agufs": (lambda: agufs_score(X, W, k=n_keep),True,),
        "udfs_like": (lambda: udfs_like_score(X, alpha=udfs_alpha, k=n_keep),True,),
        "correlation_uniqueness": (lambda: correlation_uniqueness_score(X, k=n_keep),True,),
        "mrmr": (lambda: mrmr_score(X, n_clusters=mcfs_clusters, k=n_keep),True,),
        "jmi": (lambda: jmi_score(X, n_clusters=mcfs_clusters, k=n_keep),True,),
        "cmim": (lambda: cmim_score(X, n_clusters=mcfs_clusters, k=n_keep),True,),
        "inf_fs": (lambda: inf_fs_score(X, k=n_keep),True,),
        "cluster_dispersion": (lambda: cluster_dispersion_score(X,n_clusters=mcfs_clusters,k=n_keep,),True,),
        "principal_feature_analysis": (lambda: principal_feature_analysis_score(X, k=n_keep),True,),
        "feature_cluster_representative": (lambda: feature_cluster_representative_score(X,n_feature_clusters=mcfs_clusters,k=n_keep,),True,),
    }

    # if TORCH_AVAILABLE:
    #     methods["cae"] = (lambda: cae_score(X, k_select=n_keep or 10),True,)

    rows = []
    details = {}

    iterable = methods.items()

    if show_progress:
        iterable = tqdm(
            iterable,
            desc="[3/3] Running UFS methods",
            total=len(methods),
        )

    for name, (fn, larger_better) in iterable:
        if show_progress:
            print(f"\n--> Executing: {name}")

        start = time.perf_counter()

        try:
            scores, selected_x = fn()

            elapsed = time.perf_counter() - start

            selected = _x_indices_to_columns(selected_x, groups)

            col_scores = aggregate_scores(scores, groups, larger_better)

            ordered = sorted(
                col_scores.items(),
                key=lambda x: x[1],
                reverse=larger_better,
            )

            rows.append({
                "benchmark": benchmark,
                "method": name,
                "selected_columns": selected,
                "includes_target": target_column in selected,
                "n_selected": len(selected),
                "n_noise": count_noise(selected),
                "noise_ratio": (
                    count_noise(selected) / len(selected)
                    if selected else 0
                ),
                "execution_time_sec": round(elapsed, 4),
                "error": None,
            })

            details[name] = {
                "scores": col_scores,
                "ordered": ordered,
                "execution_time_sec": elapsed,
                "error": None,
            }

            if show_progress:
                print(f"Finished {name} in {elapsed:.3f} sec")

        except Exception as e:
            elapsed = time.perf_counter() - start

            rows.append({
                "benchmark": benchmark,
                "method": name,
                "selected_columns": [],
                "includes_target": False,
                "n_selected": 0,
                "n_noise": 0,
                "noise_ratio": 0,
                "execution_time_sec": round(elapsed, 4),
                "error": str(e),
            })

            details[name] = {
                "scores": {},
                "ordered": [],
                "execution_time_sec": elapsed,
                "error": str(e),
            }

            if show_progress:
                print(f"Error in {name}: {e}")

    df_results = pd.DataFrame(rows)

    csv_path = "ufs_results.csv"

    if os.path.exists(csv_path):
        df_results.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_results.to_csv(csv_path, index=False)

    total_time = time.perf_counter() - total_start

    summary = pd.DataFrame(rows).sort_values(
        ["n_noise", "execution_time_sec"]
    ).reset_index(drop=True)

    if show_progress:
        print("\n========================================")
        print(f"Total benchmark runtime: {total_time:.3f} sec")
        print("========================================")

    details["_global_timing"] = {
        "prepare_dataframe_sec": prep_time,
        "graph_build_sec": graph_time,
        "total_runtime_sec": total_time,
    }

    return summary, details

In [18]:
import yaml
import pandas as pd

with open("../benchmarks.yaml") as f:
        benchmarks_data = yaml.safe_load(f)
        names = list(benchmarks_data.keys())

# for benchmark_name in names:
for benchmark_name in ["students", "pendigits"]:
    df = pd.read_csv(f"C:/Projects/arclo/data/noise_cf1/{benchmark_name}_cf1_noise.csv")
    benchmark_configuration = benchmarks_data[benchmark_name]
    target_column = benchmark_configuration["target_column"]

    summary, details = run_ufs_benchmark(df, benchmark_name, target_column)

    print(summary[["method", "n_selected", "n_noise", "noise_ratio", "execution_time_sec"]])
    print(details["_global_timing"])


[1/3] Preparing dataframe...
Done in 0.043 sec

[2/3] Building similarity graph...
Done in 0.476 sec


[3/3] Running UFS methods:   0%|          | 0/16 [00:00<?, ?it/s]


--> Executing: variance_score
Finished variance_score in 0.002 sec

--> Executing: laplacian_score


[3/3] Running UFS methods:  12%|█▎        | 2/16 [00:02<00:15,  1.13s/it]

Finished laplacian_score in 2.250 sec

--> Executing: spec


...

Finished jmi in 2.460 sec

--> Executing: cmim


[3/3] Running UFS methods:  88%|████████▊ | 14/16 [14:19<00:29, 14.87s/it]

Finished cmim in 2.450 sec

--> Executing: inf_fs
Finished inf_fs in 0.007 sec

--> Executing: cluster_dispersion
Finished cluster_dispersion in 0.195 sec


[3/3] Running UFS methods: 100%|██████████| 16/16 [14:19<00:00, 53.74s/it]


--> Executing: principal_feature_analysis
Finished principal_feature_analysis in 0.005 sec

--> Executing: feature_cluster_representative
Finished feature_cluster_representative in 0.089 sec

Total benchmark runtime: 862.523 sec
                            method  n_selected  n_noise  noise_ratio  \
0               cluster_dispersion           6        0     0.000000   
1                              jmi           2        0     0.000000   
2                             mrmr           2        0     0.000000   
3                             spec           3        0     0.000000   
4                              rsr           3        0     0.000000   
5                             ndfs           3        0     0.000000   
6                             mcfs          13        0     0.000000   
7       principal_feature_analysis          18        1     0.055556   
8                             cmim           2        1     0.500000   
9                        udfs_like          17    

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("ufs_results_final_final.csv")  # if needed


# drop selected_columns
df = df.drop(columns=["selected_columns"])

original_feature_counts = {
    "bank": 16,
    "default_payment": 24,
    "jannis": 54,
    "miniboone": 50,
    "appliances": 28,
    "avocado_sales": 13,
    "diamonds": 9,
    "house_sales": 17,
    "nasa": 22,
    "pol": 48,
    "superconductivity": 82,
    "covertype": 13,
    "drive_diagnosis": 48,
    "dry_beans": 17,
    "mice_protein": 81,
    "pendigits": 17,
    "students": 36
}


# Number of true selected features
df["n_true_selected"] = df["n_selected"] - df["n_noise"]

# Ratio of true selected features over total original features
df["selected_ratio"] = df.apply(
    lambda row: row["n_true_selected"] / original_feature_counts[row["benchmark"]],
    axis=1
)

# Optional: percentage instead of fraction
df["selected_ratio_pct"] = df["selected_ratio"] * 100


def confidence_interval(series, confidence=0.95):
    """
    Compute mean confidence interval using Student's t distribution.
    """
    series = series.dropna()
    n = len(series)

    if n < 2:
        return (np.nan, np.nan)

    mean = series.mean()
    sem = stats.sem(series)
    margin = sem * stats.t.ppf((1 + confidence) / 2.0, n - 1)

    return (mean - margin, mean + margin)


# Aggregate metrics
summary = df.groupby("method").agg({
    "n_selected": "mean",
    "n_noise": "mean",
    "noise_ratio": "mean",
    "execution_time_sec": "mean",
    "selected_ratio": ["mean", "min"],
    "includes_target": lambda x: (
        x.astype(str)
         .map({"True": True, "False": False})
         .astype(float)
         .mean() * 100
    )
}).reset_index()

# Flatten multi-index columns
summary.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in summary.columns
]

# Confidence intervals for selected_ratio
ci_selected = (
    df.groupby("method")["selected_ratio"]
      .apply(confidence_interval)
      .reset_index(name="ci")
)

ci_selected["selected_ratio_ci_low"] = ci_selected["ci"].str[0]
ci_selected["selected_ratio_ci_high"] = ci_selected["ci"].str[1]
ci_selected = ci_selected.drop(columns="ci")

# Confidence intervals for noise_ratio
ci_noise = (
    df.groupby("method")["noise_ratio"]
      .apply(confidence_interval)
      .reset_index(name="ci")
)

ci_noise["noise_ratio_ci_low"] = ci_noise["ci"].str[0]
ci_noise["noise_ratio_ci_high"] = ci_noise["ci"].str[1]
ci_noise = ci_noise.drop(columns="ci")

# Merge CIs into summary
summary = (
    summary
    .merge(ci_selected, on="method", how="left")
    .merge(ci_noise, on="method", how="left")
)

# Rename columns
summary = summary.rename(columns={
    "execution_time_sec_mean": "execution_time",
    "n_selected_mean": "n_selected",
    "n_noise_mean": "n_noise",
    "noise_ratio_mean": "noise_ratio",
    "selected_ratio_mean": "selected_ratio_mean",
    "selected_ratio_min": "selected_ratio_min",
    "includes_target_<lambda>": "includes_target_pct"
})

# Compute CI half-widths (margin of error)
summary["selected_ratio_ci"] = (
    summary["selected_ratio_ci_high"]
    - summary["selected_ratio_mean"]
)

summary["noise_ratio_ci"] = (
    summary["noise_ratio_ci_high"]
    - summary["noise_ratio"]
)

# Format columns as "value (± CI)"
summary["selected_ratio"] = summary.apply(
    lambda row: (
        f"{row['selected_ratio_mean']:.3f} "
        f"(± {row['selected_ratio_ci']:.3f})"
    ),
    axis=1
)

summary["noise_ratio"] = summary.apply(
    lambda row: (
        f"{row['noise_ratio']:.3f} "
        f"(± {row['noise_ratio_ci']:.3f})"
    ),
    axis=1
)

# Remove intermediate CI columns
summary = summary.drop(columns=[
    "selected_ratio_mean",
    "selected_ratio_ci_low",
    "selected_ratio_ci_high",
    "selected_ratio_ci",
    "noise_ratio_ci_low",
    "noise_ratio_ci_high",
    "noise_ratio_ci"
])

summary = summary.round(3)

summary = summary.sort_values(
    by=["noise_ratio", "n_selected"],
    ascending=[True, False]
)

summary

,method,n_selected,n_noise,noise_ratio,execution_time,selected_ratio_min,includes_target_pct,selected_ratio
9,lima,30.647,0.706,0.016 (± 0.023),11.129,0.354,100.000,0.915 (± 0.099)
2,cluster_dispersion,6.941,0.176,0.044 (± 0.068),0.419,0.037,23.529,0.273 (± 0.129)
13,principal_feature_analysis,14.000,0.412,0.053 (± 0.054),0.047,0.025,47.059,0.403 (± 0.181)
14,rsr,1.706,0.294,0.082 (± 0.093),284.857,0.000,0.000,0.050 (± 0.031)
10,mcfs,3.412,0.294,0.111 (± 0.138),198.711,0.000,17.647,0.125 (± 0.138)
11,mrmr,11.353,0.294,0.138 (± 0.172),9.879,0.000,47.059,0.351 (± 0.173)
16,udfs_like,13.471,0.765,0.172 (± 0.106),275.060,0.028,41.176,0.342 (± 0.198)
15,spec,1.647,0.529,0.206 (± 0.147),179.027,0.000,5.882,0.036 (± 0.024)
7,jmi,12.000,0.412,0.210 (± 0.184),9.845,0.000,35.294,0.358 (± 0.194)
12,ndfs,1.059,0.412,0.245 (± 0.210),176.395,0.000,5.882,0.034 (± 0.029)
